## 数据处理

### 导入相应模型

In [1]:
import gc # 垃圾回收
import pandas as pd 
import numpy as np
import warnings
warnings.filterwarnings('ignore')

#导入分析库
from sklearn.model_selection import train_test_split # 数据拆分
from sklearn.model_selection import StratifiedKFold # 同分布数据拆分，交叉验证
import lightgbm as lgb # 微软
import xgboost as xgb

### 数据加载

In [2]:
%%time
# 加载数据
# 用户行为日志
user_log = pd.read_csv('./data_format1/user_log_format1.csv', dtype={'time_stamp':'str'})
# 用户画像
user_info = pd.read_csv('./data_format1/user_info_format1.csv')
# 训练数据和测试数据
train_data = pd.read_csv('./data_format1/train_format1.csv')
test_data = pd.read_csv('./data_format1/test_format1.csv')

Wall time: 18.6 s


### 查看数据

In [3]:
print('---data shape---')     
for data in [user_log, user_info, train_data, test_data]:
    print(data.shape)

---data shape---
(54925330, 7)
(424170, 3)
(260864, 3)
(261477, 3)


In [4]:
print('---data info ---')
for data in [user_log, user_info, train_data, test_data]:
    print(data.info())

---data info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54925330 entries, 0 to 54925329
Data columns (total 7 columns):
 #   Column       Dtype  
---  ------       -----  
 0   user_id      int64  
 1   item_id      int64  
 2   cat_id       int64  
 3   seller_id    int64  
 4   brand_id     float64
 5   time_stamp   object 
 6   action_type  int64  
dtypes: float64(1), int64(5), object(1)
memory usage: 2.9+ GB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 424170 entries, 0 to 424169
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   user_id    424170 non-null  int64  
 1   age_range  421953 non-null  float64
 2   gender     417734 non-null  float64
dtypes: float64(2), int64(1)
memory usage: 9.7 MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260864 entries, 0 to 260863
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   u

In [5]:
display(user_info.head())

,user_id,age_range,gender
0,376517,6.0,1.0
1,234512,5.0,0.0
2,344532,5.0,0.0
3,186135,5.0,0.0
4,30230,5.0,0.0


In [6]:
display(train_data.head(),test_data.head())

,user_id,merchant_id,label
0,34176,3906,0
1,34176,121,0
2,34176,4356,1
3,34176,2217,0
4,230784,4818,0


,user_id,merchant_id,prob
0,163968,4605,NaN
1,360576,1581,NaN
2,98688,1964,NaN
3,98688,3645,NaN
4,295296,3361,NaN


### 数据集成

In [7]:
train_data['origin'] = 'train'
test_data['origin'] = 'test'
# 集成
all_data = pd.concat([train_data, test_data], ignore_index=True, sort=False)
# prob测试数据中特有的一列
all_data.drop(['prob'], axis=1, inplace=True) # 删除概率这一列
display(all_data.head(),all_data.shape)

,user_id,merchant_id,label,origin
0,34176,3906,0.0,train
1,34176,121,0.0,train
2,34176,4356,1.0,train
3,34176,2217,0.0,train
4,230784,4818,0.0,train


(522341, 4)

In [8]:
# 连接user_info表，通过user_id关联
all_data = all_data.merge(user_info, on='user_id', how='left')
display(all_data.shape,all_data.head())

(522341, 6)

,user_id,merchant_id,label,origin,age_range,gender
0,34176,3906,0.0,train,6.0,0.0
1,34176,121,0.0,train,6.0,0.0
2,34176,4356,1.0,train,6.0,0.0
3,34176,2217,0.0,train,6.0,0.0
4,230784,4818,0.0,train,0.0,0.0


In [9]:
# 使用 merchant_id（原列名seller_id）
user_log.rename(columns={'seller_id':'merchant_id'}, inplace=True)

In [10]:
del train_data,test_data,user_info
gc.collect()

48

### 数据类型转换

In [11]:
%%time
display(user_log.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54925330 entries, 0 to 54925329
Data columns (total 7 columns):
 #   Column       Dtype  
---  ------       -----  
 0   user_id      int64  
 1   item_id      int64  
 2   cat_id       int64  
 3   merchant_id  int64  
 4   brand_id     float64
 5   time_stamp   object 
 6   action_type  int64  
dtypes: float64(1), int64(5), object(1)
memory usage: 2.9+ GB


None

Wall time: 8.94 ms


In [12]:
%%time
display(user_log.head())

,user_id,item_id,cat_id,merchant_id,brand_id,time_stamp,action_type
0,328862,323294,833,2882,2661.0,0829,0
1,328862,844400,1271,2882,2661.0,0829,0
2,328862,575153,1271,2882,2661.0,0829,0
3,328862,996875,1271,2882,2661.0,0829,0
4,328862,1086186,1271,1253,1049.0,0829,0


Wall time: 7.99 ms


In [13]:
%%time
# 用户行为数据类型转换
user_log['user_id'] = user_log['user_id'].astype('int32')
user_log['merchant_id'] = user_log['merchant_id'].astype('int32')
user_log['item_id'] = user_log['item_id'].astype('int32')
user_log['cat_id'] = user_log['cat_id'].astype('int32')
user_log['brand_id'].fillna(0, inplace=True)
user_log['brand_id'] = user_log['brand_id'].astype('int32')
user_log['time_stamp'] = pd.to_datetime(user_log['time_stamp'], format='%H%M')
user_log['action_type'] = user_log['action_type'].astype('int32')
display(user_log.info(),user_log.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54925330 entries, 0 to 54925329
Data columns (total 7 columns):
 #   Column       Dtype         
---  ------       -----         
 0   user_id      int32         
 1   item_id      int32         
 2   cat_id       int32         
 3   merchant_id  int32         
 4   brand_id     int32         
 5   time_stamp   datetime64[ns]
 6   action_type  int32         
dtypes: datetime64[ns](1), int32(6)
memory usage: 1.6 GB


None

,user_id,item_id,cat_id,merchant_id,brand_id,time_stamp,action_type
0,328862,323294,833,2882,2661,1900-01-01 08:29:00,0
1,328862,844400,1271,2882,2661,1900-01-01 08:29:00,0
2,328862,575153,1271,2882,2661,1900-01-01 08:29:00,0
3,328862,996875,1271,2882,2661,1900-01-01 08:29:00,0
4,328862,1086186,1271,1253,1049,1900-01-01 08:29:00,0


Wall time: 8.59 s


In [14]:
display(all_data.isnull().sum())

user_id             0
merchant_id         0
label          261477
origin              0
age_range        2578
gender           7545
dtype: int64

In [15]:
# 缺失值填充
all_data['age_range'].fillna(0, inplace=True)
all_data['gender'].fillna(2, inplace=True)
all_data.isnull().sum()

user_id             0
merchant_id         0
label          261477
origin              0
age_range           0
gender              0
dtype: int64

In [16]:
all_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 522341 entries, 0 to 522340
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   user_id      522341 non-null  int64  
 1   merchant_id  522341 non-null  int64  
 2   label        260864 non-null  float64
 3   origin       522341 non-null  object 
 4   age_range    522341 non-null  float64
 5   gender       522341 non-null  float64
dtypes: float64(3), int64(2), object(1)
memory usage: 27.9+ MB


In [17]:
2**8
# -128 ~ 127

256

In [18]:
2**32

4294967296

In [19]:
all_data['age_range'] = all_data['age_range'].astype('int8')
all_data['gender'] = all_data['gender'].astype('int8')
all_data['label'] = all_data['label'].astype('str')
all_data['user_id'] = all_data['user_id'].astype('int32')
all_data['merchant_id'] = all_data['merchant_id'].astype('int32')
all_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 522341 entries, 0 to 522340
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   user_id      522341 non-null  int32 
 1   merchant_id  522341 non-null  int32 
 2   label        522341 non-null  object
 3   origin       522341 non-null  object
 4   age_range    522341 non-null  int8  
 5   gender       522341 non-null  int8  
dtypes: int32(2), int8(2), object(2)
memory usage: 16.9+ MB


### 用户特征工程(5min)

In [20]:
%%time
##### 特征处理
##### User特征处理
groups = user_log.groupby(['user_id'])

# 用户交互行为数量 u1
temp = groups.size().reset_index().rename(columns={0:'u1'})
all_data = all_data.merge(temp, on='user_id', how='left')

# 细分
# 使用 agg 基于列的聚合操作，统计唯一值个数 item_id, cat_id, merchant_id, brand_id
# 用户，交互行为：点了多少商品呢？
temp = groups['item_id'].agg([('u2', 'nunique')]).reset_index()
all_data = all_data.merge(temp, on='user_id', how='left')

# 用户，交互行为，具体统计：类目多少
temp = groups['cat_id'].agg([('u3', 'nunique')]).reset_index()
all_data = all_data.merge(temp, on='user_id', how='left')

temp = groups['merchant_id'].agg([('u4', 'nunique')]).reset_index()
all_data = all_data.merge(temp, on='user_id', how='left')

temp = groups['brand_id'].agg([('u5', 'nunique')]).reset_index()
all_data = all_data.merge(temp, on='user_id', how='left')


# 购物时间间隔特征 u6 按照小时
temp = groups['time_stamp'].agg([('F_time', 'min'), ('B_time', 'max')]).reset_index()
temp['u6'] = (temp['B_time'] - temp['F_time']).dt.seconds/3600
all_data = all_data.merge(temp[['user_id', 'u6']], on='user_id', how='left')


# 统计操作类型为0，1，2，3的个数
temp = groups['action_type'].value_counts().unstack().reset_index().rename(
    columns={0:'u7', 1:'u8', 2:'u9', 3:'u10'})
all_data = all_data.merge(temp, on='user_id', how='left')

del temp,groups
gc.collect()

Wall time: 4min 18s


16279

In [21]:
all_data.head()

,user_id,merchant_id,label,origin,age_range,gender,u1,u2,u3,u4,u5,u6,u7,u8,u9,u10
0,34176,3906,0.0,train,6,0,451,256,45,109,108,5.833333,410.0,NaN,34.0,7.0
1,34176,121,0.0,train,6,0,451,256,45,109,108,5.833333,410.0,NaN,34.0,7.0
2,34176,4356,1.0,train,6,0,451,256,45,109,108,5.833333,410.0,NaN,34.0,7.0
3,34176,2217,0.0,train,6,0,451,256,45,109,108,5.833333,410.0,NaN,34.0,7.0
4,230784,4818,0.0,train,0,0,54,31,17,20,19,5.166667,47.0,NaN,7.0,NaN


### 店铺特征工程(5min)

In [22]:
%%time
##### 商家特征处理
groups = user_log.groupby(['merchant_id'])

# 商家被交互行为数量 m1
temp = groups.size().reset_index().rename(columns={0:'m1'})
all_data = all_data.merge(temp, on='merchant_id', how='left')

# 统计商家被交互的 user_id, item_id, cat_id, brand_id 唯一值
temp = groups['user_id', 'item_id', 'cat_id', 'brand_id'].nunique().reset_index().rename(
    columns={
    'user_id':'m2',
    'item_id':'m3', 
    'cat_id':'m4', 
    'brand_id':'m5'})
all_data = all_data.merge(temp, on='merchant_id', how='left')

# 统计商家被交互的 action_type 唯一值
temp = groups['action_type'].value_counts().unstack().reset_index().rename(  
    columns={0:'m6', 1:'m7', 2:'m8', 3:'m9'})
all_data = all_data.merge(temp, on='merchant_id', how='left')

del temp
gc.collect()

Wall time: 5min 21s


8798

In [23]:
display(all_data.tail())

,user_id,merchant_id,label,origin,age_range,gender,u1,u2,u3,u4,...,u10,m1,m2,m3,m4,m5,m6,m7,m8,m9
522336,228479,3111,nan,test,6,0,2004,1173,71,278,...,208.0,10105,4154,542,50,18,8997.0,9.0,687.0,412.0
522337,97919,2341,nan,test,8,1,55,29,14,17,...,1.0,5543,1592,352,93,19,4548.0,6.0,815.0,174.0
522338,97919,3971,nan,test,8,1,55,29,14,17,...,1.0,28892,7587,272,7,2,24602.0,94.0,2608.0,1588.0
522339,32639,3536,nan,test,0,0,72,46,24,33,...,1.0,14027,4956,322,19,3,12807.0,29.0,793.0,398.0
522340,32639,3319,nan,test,0,0,72,46,24,33,...,1.0,25959,7927,952,175,85,21737.0,34.0,2700.0,1488.0


### 用户和店铺联合特征工程(4min)

In [24]:
%%time
##### 用户+商户特征
groups = user_log.groupby(['user_id', 'merchant_id'])

# 用户在不同商家交互统计
temp = groups.size().reset_index().rename(columns={0:'um1'})
all_data = all_data.merge(temp, on=['user_id', 'merchant_id'], how='left')

# 统计用户在不同商家交互的 item_id, cat_id, brand_id 唯一值
temp = groups['item_id', 'cat_id', 'brand_id'].nunique().reset_index().rename(
    columns={
    'item_id':'um2',
    'cat_id':'um3',
    'brand_id':'um4'})
all_data = all_data.merge(temp, on=['user_id', 'merchant_id'], how='left')

# 统计用户在不同商家交互的 action_type 唯一值
temp = groups['action_type'].value_counts().unstack().reset_index().rename(
    columns={
    0:'um5',
    1:'um6',
    2:'um7',
    3:'um8'})
all_data = all_data.merge(temp, on=['user_id', 'merchant_id'], how='left')

# 统计用户在不同商家购物时间间隔特征 um9 按照小时
temp = groups['time_stamp'].agg([('F_time', 'min'), ('B_time', 'max')]).reset_index()
temp['um9'] = (temp['B_time'] - temp['F_time']).dt.seconds/3600
all_data = all_data.merge(temp[['user_id','merchant_id','um9']], on=['user_id', 'merchant_id'], how='left')

del temp,groups
gc.collect()

Wall time: 4min 22s


9096

In [25]:
display(all_data.head())

,user_id,merchant_id,label,origin,age_range,gender,u1,u2,u3,u4,...,m9,um1,um2,um3,um4,um5,um6,um7,um8,um9
0,34176,3906,0.0,train,6,0,451,256,45,109,...,961.0,39,20,6,1,36.0,NaN,1.0,2.0,0.850000
1,34176,121,0.0,train,6,0,451,256,45,109,...,2699.0,14,1,1,1,13.0,NaN,1.0,NaN,0.050000
2,34176,4356,1.0,train,6,0,451,256,45,109,...,196.0,18,2,1,1,12.0,NaN,6.0,NaN,0.016667
3,34176,2217,0.0,train,6,0,451,256,45,109,...,4150.0,2,1,1,1,1.0,NaN,1.0,NaN,0.000000
4,230784,4818,0.0,train,0,0,54,31,17,20,...,1959.0,8,1,1,1,7.0,NaN,1.0,NaN,0.050000


### 购买点击比

In [26]:
all_data['r1'] = all_data['u9']/all_data['u7'] # 用户购买点击比
all_data['r2'] = all_data['m8']/all_data['m6'] # 商家购买点击比
all_data['r3'] = all_data['um7']/all_data['um5'] #不同用户不同商家购买点击比
display(all_data.head())

,user_id,merchant_id,label,origin,age_range,gender,u1,u2,u3,u4,...,um3,um4,um5,um6,um7,um8,um9,r1,r2,r3
0,34176,3906,0.0,train,6,0,451,256,45,109,...,6,1,36.0,NaN,1.0,2.0,0.850000,0.082927,0.027572,0.027778
1,34176,121,0.0,train,6,0,451,256,45,109,...,1,1,13.0,NaN,1.0,NaN,0.050000,0.082927,0.066145,0.076923
2,34176,4356,1.0,train,6,0,451,256,45,109,...,1,1,12.0,NaN,6.0,NaN,0.016667,0.082927,0.158024,0.500000
3,34176,2217,0.0,train,6,0,451,256,45,109,...,1,1,1.0,NaN,1.0,NaN,0.000000,0.082927,0.071243,1.000000
4,230784,4818,0.0,train,0,0,54,31,17,20,...,1,1,7.0,NaN,1.0,NaN,0.050000,0.148936,0.063164,0.142857


### 空数据填充

In [27]:
display(all_data.isnull().sum())

user_id             0
merchant_id         0
label               0
origin              0
age_range           0
gender              0
u1                  0
u2                  0
u3                  0
u4                  0
u5                  0
u6                  0
u7                360
u8             484162
u9                  0
u10            227482
m1                  0
m2                  0
m3                  0
m4                  0
m5                  0
m6                  0
m7               4052
m8                  0
m9                  0
um1                 0
um2                 0
um3                 0
um4                 0
um5             59408
um6            512947
um7                 0
um8            425790
um9                 0
r1                360
r2                  0
r3              59408
dtype: int64

In [28]:
all_data.fillna(0, inplace=True)

In [29]:
all_data.isnull().sum()

user_id        0
merchant_id    0
label          0
origin         0
age_range      0
gender         0
u1             0
u2             0
u3             0
u4             0
u5             0
u6             0
u7             0
u8             0
u9             0
u10            0
m1             0
m2             0
m3             0
m4             0
m5             0
m6             0
m7             0
m8             0
m9             0
um1            0
um2            0
um3            0
um4            0
um5            0
um6            0
um7            0
um8            0
um9            0
r1             0
r2             0
r3             0
dtype: int64

### 年龄性别类别型转换

In [31]:
all_data['age_range']

0         6
1         6
2         6
3         6
4         0
         ..
522336    6
522337    8
522338    8
522339    0
522340    0
Name: age_range, Length: 522341, dtype: int8

In [30]:
%%time
# 修改age_range字段名称为 age_0, age_1, age_2... age_8
# 独立编码
temp = pd.get_dummies(all_data['age_range'], prefix='age')
display(temp.head(10))
all_data = pd.concat([all_data, temp], axis=1)

,age_0,age_1,age_2,age_3,age_4,age_5,age_6,age_7,age_8
0,0,0,0,0,0,0,1,0,0
1,0,0,0,0,0,0,1,0,0
2,0,0,0,0,0,0,1,0,0
3,0,0,0,0,0,0,1,0,0
4,1,0,0,0,0,0,0,0,0
5,0,0,0,0,1,0,0,0,0
6,0,0,0,0,0,1,0,0,0
7,0,0,0,0,0,1,0,0,0
8,0,0,0,0,0,1,0,0,0
9,0,0,0,0,1,0,0,0,0


Wall time: 125 ms


In [32]:
# 性别转换
temp = pd.get_dummies(all_data['gender'], prefix='g')
all_data = pd.concat([all_data, temp], axis=1) # 列进行合并

# 删除原数据
all_data.drop(['age_range', 'gender'], axis=1, inplace=True)

del temp
gc.collect()

18438

In [33]:
all_data.head()

,user_id,merchant_id,label,origin,u1,u2,u3,u4,u5,u6,...,age_2,age_3,age_4,age_5,age_6,age_7,age_8,g_0,g_1,g_2
0,34176,3906,0.0,train,451,256,45,109,108,5.833333,...,0,0,0,0,1,0,0,1,0,0
1,34176,121,0.0,train,451,256,45,109,108,5.833333,...,0,0,0,0,1,0,0,1,0,0
2,34176,4356,1.0,train,451,256,45,109,108,5.833333,...,0,0,0,0,1,0,0,1,0,0
3,34176,2217,0.0,train,451,256,45,109,108,5.833333,...,0,0,0,0,1,0,0,1,0,0
4,230784,4818,0.0,train,54,31,17,20,19,5.166667,...,0,0,0,0,0,0,0,1,0,0


### 数据存储

In [34]:
%%time
# train_data、test-data
train_data = all_data[all_data['origin'] == 'train'].drop(['origin'], axis=1)
test_data = all_data[all_data['origin'] == 'test'].drop(['label', 'origin'], axis=1)

train_data.to_csv('train_data.csv')
test_data.to_csv('test_data.csv')

Wall time: 9.14 s


## 算法建模预测

In [35]:
# 训练数据和目标值
train_X, train_y = train_data.drop(['label'], axis=1), train_data['label']

# 数据拆分保留20%作为测试数据
X_train, X_valid, y_train, y_valid = train_test_split(train_X, train_y, test_size=.2)

### LGB 模型

In [36]:
def lgb_train(X_train, y_train, X_valid, y_valid, verbose=True):
    model_lgb = lgb.LGBMClassifier(
        max_depth=10, # 8 # 树最大的深度
        n_estimators=5000, # 集成算法，树数量
        min_child_weight=100, 
        colsample_bytree=0.7, # 特征筛选
        subsample=0.9,  # 样本采样比例
        learning_rate=0.1) # 学习率

    model_lgb.fit(
        X_train, 
        y_train,
        eval_metric='auc',
        eval_set=[(X_train, y_train), (X_valid, y_valid)],
        verbose=verbose, # 是否打印输出训练过程
        early_stopping_rounds=10) # 早停，等10轮决策，评价指标不在变化，停止

    print(model_lgb.best_score_['valid_1']['auc'])
    return model_lgb

In [39]:
X_train

,user_id,merchant_id,u1,u2,u3,u4,u5,u6,u7,u8,...,age_2,age_3,age_4,age_5,age_6,age_7,age_8,g_0,g_1,g_2
132344,210694,2928,11,9,6,5,5,3.150000,10.0,0.0,...,0,0,1,0,0,0,0,1,0,0
19029,187320,3484,57,34,17,21,22,4.883333,46.0,0.0,...,0,0,0,0,0,0,0,0,1,0
79580,345450,3350,67,46,9,17,17,5.916667,58.0,0.0,...,0,0,0,0,0,0,0,1,0,0
99577,69541,3629,47,30,16,21,24,5.816667,39.0,0.0,...,0,0,0,0,0,0,0,0,1,0
67041,214341,2856,12,7,4,7,7,5.100000,7.0,0.0,...,0,1,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126273,78580,2895,424,224,47,82,90,4.100000,344.0,0.0,...,0,1,0,0,0,0,0,1,0,0
4207,263820,1910,21,16,8,6,6,4.983333,19.0,0.0,...,0,0,0,0,0,0,0,0,1,0
143047,230182,3123,29,18,15,13,13,5.983333,14.0,0.0,...,0,1,0,0,0,0,0,1,0,0
156183,86092,4044,165,106,38,65,68,5.150000,150.0,0.0,...,0,0,0,0,0,0,0,1,0,0


In [40]:
X_train.values

array([[2.10694e+05, 2.92800e+03, 1.10000e+01, ..., 1.00000e+00,
        0.00000e+00, 0.00000e+00],
       [1.87320e+05, 3.48400e+03, 5.70000e+01, ..., 0.00000e+00,
        1.00000e+00, 0.00000e+00],
       [3.45450e+05, 3.35000e+03, 6.70000e+01, ..., 1.00000e+00,
        0.00000e+00, 0.00000e+00],
       ...,
       [2.30182e+05, 3.12300e+03, 2.90000e+01, ..., 1.00000e+00,
        0.00000e+00, 0.00000e+00],
       [8.60920e+04, 4.04400e+03, 1.65000e+02, ..., 1.00000e+00,
        0.00000e+00, 0.00000e+00],
       [1.06327e+05, 1.49900e+03, 2.30000e+02, ..., 1.00000e+00,
        0.00000e+00, 0.00000e+00]])

In [38]:
model_lgb = lgb_train(X_train.values, y_train, X_valid.values, y_valid, verbose=True)

[1]	training's auc: 0.640009	training's binary_logloss: 0.228212	valid_1's auc: 0.627955	valid_1's binary_logloss: 0.229246
Training until validation scores don't improve for 10 rounds.
[2]	training's auc: 0.648943	training's binary_logloss: 0.226955	valid_1's auc: 0.636741	valid_1's binary_logloss: 0.228055
[3]	training's auc: 0.654258	training's binary_logloss: 0.225831	valid_1's auc: 0.639618	valid_1's binary_logloss: 0.227076
[4]	training's auc: 0.656223	training's binary_logloss: 0.22493	valid_1's auc: 0.641308	valid_1's binary_logloss: 0.226275
[5]	training's auc: 0.657013	training's binary_logloss: 0.224163	valid_1's auc: 0.642213	valid_1's binary_logloss: 0.225582
[6]	training's auc: 0.659946	training's binary_logloss: 0.223479	valid_1's auc: 0.644374	valid_1's binary_logloss: 0.224994
[7]	training's auc: 0.661228	training's binary_logloss: 0.222846	valid_1's auc: 0.644595	valid_1's binary_logloss: 0.224503
[8]	training's auc: 0.663428	training's binary_logloss: 0.222264	valid_

[66]	training's auc: 0.715136	training's binary_logloss: 0.212228	valid_1's auc: 0.673082	valid_1's binary_logloss: 0.218789
[67]	training's auc: 0.715597	training's binary_logloss: 0.212156	valid_1's auc: 0.673082	valid_1's binary_logloss: 0.218783
[68]	training's auc: 0.716101	training's binary_logloss: 0.212053	valid_1's auc: 0.67344	valid_1's binary_logloss: 0.218737
[69]	training's auc: 0.716458	training's binary_logloss: 0.211986	valid_1's auc: 0.673405	valid_1's binary_logloss: 0.218748
[70]	training's auc: 0.716961	training's binary_logloss: 0.211888	valid_1's auc: 0.673533	valid_1's binary_logloss: 0.218733
[71]	training's auc: 0.717296	training's binary_logloss: 0.211821	valid_1's auc: 0.673543	valid_1's binary_logloss: 0.218736
[72]	training's auc: 0.717596	training's binary_logloss: 0.21176	valid_1's auc: 0.673506	valid_1's binary_logloss: 0.218735
[73]	training's auc: 0.718504	training's binary_logloss: 0.211627	valid_1's auc: 0.673942	valid_1's binary_logloss: 0.218683
[7

In [41]:
%%time
prob = model_lgb.predict_proba(test_data.values) # 预测

submission = pd.read_csv('./data_format1/test_format1.csv')

# 复购的概率
submission['prob'] = pd.Series(prob[:,1]) # 预测数据赋值给提交数据

display(submission.head())

submission.to_csv('submission_lgb.csv', index=False)

del submission
gc.collect()

,user_id,merchant_id,prob
0,163968,4605,0.051875
1,360576,1581,0.115620
2,98688,1964,0.050633
3,98688,3645,0.032896
4,295296,3361,0.066129


Wall time: 1.53 s


22630

In [42]:
prob

array([[0.94812479, 0.05187521],
       [0.88438038, 0.11561962],
       [0.94936734, 0.05063266],
       ...,
       [0.85627925, 0.14372075],
       [0.95222606, 0.04777394],
       [0.92295125, 0.07704875]])

### XGB 模型

In [43]:
def xgb_train(X_train, y_train, X_valid, y_valid, verbose=True):
    model_xgb = xgb.XGBClassifier(
        max_depth=10, # raw8
        n_estimators=5000,
        min_child_weight=300, 
        colsample_bytree=0.7, 
        subsample=0.9, 
        learing_rate=0.1)
    
    model_xgb.fit(
        X_train, 
        y_train,
        eval_metric='auc',
        eval_set=[(X_train, y_train), (X_valid, y_valid)],
        verbose=verbose,
        early_stopping_rounds=10)# 早停法，如果auc在10epoch没有进步就stop
    print(model_xgb.best_score)
    return model_xgb

模型训练

In [44]:
model_xgb = xgb_train(X_train, y_train, X_valid, y_valid, verbose=False)

[19:20:55] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:576: 
Parameters: { "learing_rate" } might not be used.

  This could be a false alarm, with some parameters getting used by language bindings but
  then being mistakenly passed down to XGBoost core, or some parameter actually being used
  but getting flagged wrongly here. Please open an issue if you find any such cases.


0.673734


模型预测

In [45]:
%%time
prob = model_xgb.predict_proba(test_data)
submission = pd.read_csv('./data_format1/test_format1.csv')
submission['prob'] = pd.Series(prob[:,1])
submission.to_csv('submission_xgb.csv', index=False)
display(submission.head())
del submission
gc.collect()

,user_id,merchant_id,prob
0,163968,4605,0.065637
1,360576,1581,0.091968
2,98688,1964,0.058792
3,98688,3645,0.053258
4,295296,3361,0.081707


Wall time: 980 ms


509

## 交叉验证多轮建模

In [62]:
# 构造训练集和测试集
def get_train_test_datas(train_df,label_df):
    skv = StratifiedKFold(n_splits=10, shuffle=True)
    trainX = []
    trainY = []
    testX = []
    testY = []
    # 索引：训练数据索引train_index,目标值的索引test_index
    for train_index, test_index in skv.split(X=train_df, y=label_df):# 10轮for循环
        
        train_x, train_y, test_x, test_y = train_df.iloc[train_index, :], label_df.iloc[train_index], \
                                            train_df.iloc[test_index, :], label_df.iloc[test_index]

        trainX.append(train_x)
        trainY.append(train_y)
        testX.append(test_x)
        testY.append(test_y)
    return trainX, testX, trainY, testY

### LGB 模型（1min）

In [63]:
%%time
train_X, train_y = train_data.drop(['label'], axis=1), train_data['label']

# 拆分为10份训练数据和验证数据
X_train, X_valid, y_train, y_valid = get_train_test_datas(train_X, train_y)

print('----训练数据，长度',len(X_train))
print('----验证数据，长度',len(X_valid))

pred_lgbms = [] # 列表，接受目标值，10轮，平均值

for i in range(10):
    print('\n============================LGB training use Data {}/10============================\n'.format(i+1))
    model_lgb = lgb.LGBMClassifier(
        max_depth=10, # 8
        n_estimators=1000,
        min_child_weight=100,
        colsample_bytree=0.7,
        subsample=0.9,
        learning_rate=0.05)

    model_lgb.fit(
        X_train[i].values, 
        y_train[i],
        eval_metric='auc',
        eval_set=[(X_train[i].values, y_train[i]), (X_valid[i].values, y_valid[i])],
        verbose=False,
        early_stopping_rounds=10)

    print(model_lgb.best_score_['valid_1']['auc'])

    pred = model_lgb.predict_proba(test_data.values)
    
    pred = pd.DataFrame(pred[:,1]) # 将预测概率（复购）去处理，转换成DataFrame
    
    pred_lgbms.append(pred)

# 求10轮平均值生成预测结果，保存
# 每一轮的结果，作为一列，进行了添加
pred_lgbms = pd.concat(pred_lgbms, axis=1) # 级联，列进行级联

# 加载提交数据
submission = pd.read_csv('./data_format1/test_format1.csv')

submission['prob'] = pred_lgbms.mean(axis=1) # 10轮训练的平均值

submission.to_csv('submission_KFold_lgb.csv', index=False)

----训练数据，长度 10
----验证数据，长度 10

============================LGB training use Data 1/10============================

0.6764325578514025

============================LGB training use Data 2/10============================

0.6891014761649508

============================LGB training use Data 3/10============================

0.6847981034207339

============================LGB training use Data 4/10============================

0.6740686492270855

============================LGB training use Data 5/10============================

0.6800922711164193

============================LGB training use Data 6/10============================

0.6866774295822827

============================LGB training use Data 7/10============================

0.6865939749602854

============================LGB training use Data 8/10============================

0.663655985712364

============================LGB training use Data 9/10============================

0.6850513956313553

============================LGB tr

In [55]:
pred_lgbms

,0,0,0,0,0,0,0,0,0,0
0,0.055886,0.046910,0.048247,0.044899,0.051961,0.055180,0.051898,0.052081,0.052636,0.051678
1,0.092411,0.103658,0.098715,0.092163,0.095193,0.097384,0.070938,0.086493,0.105066,0.089608
2,0.049865,0.046393,0.050722,0.054663,0.053192,0.046511,0.053459,0.046542,0.053645,0.042156
3,0.028297,0.030072,0.033617,0.027951,0.032181,0.029938,0.035848,0.038321,0.033284,0.029815
4,0.070862,0.064259,0.068224,0.070074,0.069326,0.059107,0.072940,0.077195,0.062571,0.069239
...,...,...,...,...,...,...,...,...,...,...
261472,0.054240,0.057529,0.061744,0.051099,0.056373,0.044646,0.061189,0.045183,0.061670,0.062057
261473,0.026819,0.021841,0.026320,0.028997,0.027158,0.029767,0.023243,0.023721,0.025249,0.034149
261474,0.115838,0.134412,0.120677,0.117268,0.112812,0.113701,0.126516,0.126943,0.105635,0.118007
261475,0.042335,0.034865,0.040677,0.037834,0.037391,0.036060,0.037305,0.044105,0.048137,0.041567


### XGB 模型（4min）

In [56]:
# 构造训练集和测试集
def get_train_test_datas(train_df,label_df):
    skv = StratifiedKFold(n_splits=20, shuffle=True)
    trainX = []
    trainY = []
    testX = []
    testY = []
    # 索引：训练数据索引train_index,目标值的索引test_index
    for train_index, test_index in skv.split(X=train_df, y=label_df):# 10轮for循环
        
        train_x, train_y, test_x, test_y = train_df.iloc[train_index, :], label_df.iloc[train_index], \
                                            train_df.iloc[test_index, :], label_df.iloc[test_index]

        trainX.append(train_x)
        trainY.append(train_y)
        testX.append(test_x)
        testY.append(test_y)
    return trainX, testX, trainY, testY

In [57]:
%%time
train_X, train_y = train_data.drop(['label'], axis=1), train_data['label']

# 拆分为20份训练数据和验证数据
X_train, X_valid, y_train, y_valid = get_train_test_datas(train_X, train_y)

print('------数据长度',len(X_train),len(y_train))

pred_xgbs = []
for i in range(20):
    print('\n============================XGB training use Data {}/20============================\n'.format(i+1))
    model_xgb = xgb.XGBClassifier(
        max_depth=10, # raw8
        n_estimators=5000,
        min_child_weight=200, 
        colsample_bytree=0.7, 
        subsample=0.9,
        learning_rate = 0.1)

    model_xgb.fit(
        X_train[i], 
        y_train[i],
        eval_metric='auc',
        eval_set=[(X_train[i], y_train[i]), (X_valid[i], y_valid[i])],
        verbose=False,
        early_stopping_rounds=10 # 早停法，如果auc在10epoch没有进步就stop
    )    

    print(model_xgb.best_score)

    pred = model_xgb.predict_proba(test_data)
    pred = pd.DataFrame(pred[:,1])
    pred_xgbs.append(pred)

# 求20轮平均值生成预测结果，保存
pred_xgbs = pd.concat(pred_xgbs, axis=1)
submission = pd.read_csv('./data_format1/test_format1.csv')
submission['prob'] = pred_xgbs.mean(axis=1)
submission.to_csv('submission_KFold_xgb.csv', index=False)

------数据长度 20 20

============================XGB training use Data 1/20============================

0.710332

============================XGB training use Data 2/20============================

0.679258

============================XGB training use Data 3/20============================

0.70283

============================XGB training use Data 4/20============================

0.681381

============================XGB training use Data 5/20============================

0.688876

============================XGB training use Data 6/20============================

0.689305

============================XGB training use Data 7/20============================

0.674339

============================XGB training use Data 8/20============================

0.662421

============================XGB training use Data 9/20============================

0.684782

============================XGB training use Data 10/20============================

0.658125

============================XGB training u